In [8]:
import pandas as pd
import json
import os
import sys
from tqdm import tqdm
import bm25s

sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())

from scripts.config import (
    DATA_DIR,
    BM25_COLLECTIONS,
)
from scripts.retrieval_utils import (
    BM25Retriever,
    find_gold_ranks,
    get_existing_ids,
    evaluate_and_print_metrics,
    perform_retrieval,
)
from scripts.utils.json_reading import read_jsonl_basic
from scripts.utils.questions import create_questions_dataset

# Paths
QUESTIONS_DIR = os.path.join(DATA_DIR, "questions")  # NOTE: 'questions', not 'questions_singlehop'
OUTPUT_DIR = os.path.join(DATA_DIR, "retrieval")

# Question datasets configuration (same as notebook 72)
QUESTION_DATASETS = [
    {
        'name': 'Generic_LeGaulois_Debats',
        'file': 'MH_Generic_legaulois_v1.jsonl',
        'column_mapping': {
            'id_1': 'journal_id',
            'id_2': 'debat_id',
            'doc_1': 'journal_doc',
            'doc_2': 'debat_doc',
            'metadata_1': 'journal_metadata',
            'metadata_2': 'debat_metadata'
        },
        'source1_name': 'Le Gaulois',
        'source2_name': 'Les Débats',
        'question_type': 'MH - Generic'
    },
    {
        'name': 'Generic_LIntransigeant_Debats',
        'file': 'generated_questions_lintransigeant_all.jsonl',
        'column_mapping': {
            'id_1': 'journal_id',
            'id_2': 'debat_id',
            'doc_1': 'journal_doc',
            'doc_2': 'debat_doc',
            'metadata_1': 'journal_metadata',
            'metadata_2': 'debat_metadata'
        },
        'source1_name': "L'Intransigeant",
        'source2_name': 'Les Débats',
        'question_type': 'MH - Generic'
    },
    {
        'name': 'BridgeEntity_LIntransigeant_Debats',
        'file': 'MH_BridgeEntity_lintransigeant_v2.jsonl',
        'column_mapping': {
            'id_1': 'journal_id',
            'id_2': 'debat_id',
            'doc_1': 'journal_doc',
            'doc_2': 'debat_doc',
            'metadata_1': 'journal_metadata',
            'metadata_2': 'debat_metadata'
        },
        'source1_name': "L'Intransigeant",
        'source2_name': 'Les Débats',
        'question_type': 'MH - BridgeEntity'
    },
    {
        'name': 'Comparative_LeGaulois_LIntransigeant',
        'file': 'MH_Comparative_legaulois_lintransigeant_v3.jsonl',
        'column_mapping': {
            'id_1': 'legaulois_id',
            'id_2': 'lintransigeant_id',
            'doc_1': 'legaulois_doc',
            'doc_2': 'lintransigeant_doc',
            'metadata_1': 'legaulois_metadata',
            'metadata_2': 'lintransigeant_metadata'
        },
        'source1_name': 'Le Gaulois',
        'source2_name': "L'Intransigeant",
        'question_type': 'MH - Comparative'
    },
    {
        'name': 'Generic_LeGaulois_LIntransigeant',
        'file': 'MH_Generic_legaulois_lintransigeant_v3.jsonl',
        'column_mapping': {
            'id_1': 'legaulois_id',
            'id_2': 'lintransigeant_id',
            'doc_1': 'legaulois_doc',
            'doc_2': 'lintransigeant_doc',
            'metadata_1': 'legaulois_metadata',
            'metadata_2': 'lintransigeant_metadata'
        },
        'source1_name': 'Le Gaulois',
        'source2_name': "L'Intransigeant",
        'question_type': 'MH - Generic'
    }
]

N_RESULTS = 10
K_VALUES = [1, 2, 3, 5, 10]


In [2]:
# ========================================================================
# Step 1: Load all BM25 indexes
# ========================================================================
print("Step 1: Loading BM25 indexes...")

bm25_retrievers = {}
for label, index_dir in BM25_COLLECTIONS.items():
    if not os.path.exists(index_dir):
        print(f"  WARNING: BM25 index not found at {index_dir} — skipping '{label}'")
        continue
    retriever_bm25 = bm25s.BM25.load(index_dir, load_corpus=True)
    bm25_retrievers[label] = retriever_bm25
    print(f"  Loaded '{label}': {len(retriever_bm25.corpus)} documents")

retriever = BM25Retriever(bm25_retrievers, n_results=N_RESULTS)
print(f"\nBM25Retriever ready with {len(bm25_retrievers)} collections")


Step 1: Loading BM25 indexes...
  Loaded 'Les Débats': 5103 documents
  Loaded 'L'Intransigeant': 79 documents
  Loaded 'Le Gaulois': 78 documents

BM25Retriever ready with 3 collections


In [3]:
# ========================================================================
# Step 2: Load and prepare all question datasets
# ========================================================================
print("Step 2: Loading question datasets...")

all_questions = []

for dataset_config in QUESTION_DATASETS:
    print(f"\n  Processing: {dataset_config['name']}")
    jsonl_path = os.path.join(QUESTIONS_DIR, dataset_config['file'])

    if not os.path.exists(jsonl_path):
        print(f"    WARNING: File not found - {jsonl_path}")
        continue

    df_questions = create_questions_dataset(
        jsonl_path=jsonl_path,
        column_mapping=dataset_config['column_mapping'],
        source1_name=dataset_config['source1_name'],
        source2_name=dataset_config['source2_name'],
        question_type=dataset_config['question_type'],
    )
    all_questions.append(df_questions)
    print(f"    Loaded {len(df_questions)} questions")

df = pd.concat(all_questions, ignore_index=True)
print(f"\n  Total questions loaded: {len(df)}")
print(f"  Question types: {df['question_type'].value_counts().to_dict()}")

Step 2: Loading question datasets...

  Processing: Generic_LeGaulois_Debats
Created dataset with 220 questions from 77 original entries
    Loaded 220 questions

  Processing: Generic_LIntransigeant_Debats
Created dataset with 199 questions from 79 original entries
    Loaded 199 questions

  Processing: BridgeEntity_LIntransigeant_Debats
Created dataset with 152 questions from 79 original entries
    Loaded 152 questions

  Processing: Comparative_LeGaulois_LIntransigeant
Created dataset with 192 questions from 47 original entries
    Loaded 192 questions

  Processing: Generic_LeGaulois_LIntransigeant
Created dataset with 122 questions from 48 original entries
    Loaded 122 questions

  Total questions loaded: 885
  Question types: {'MH - Generic': 541, 'MH - Comparative': 192, 'MH - BridgeEntity': 152}


In [4]:
# ========================================================================
# Step 3: Perform BM25 retrieval
# ========================================================================
print("Step 3: Performing BM25 retrieval...")

retrieval_results_file = os.path.join(OUTPUT_DIR, "MH_bm25_results.jsonl")

perform_retrieval(df, retriever, output_file=retrieval_results_file)


Step 3: Performing BM25 retrieval...
Found 0 existing retrieval results
Retrieving for 885 questions...


100%|██████████| 885/885 [00:28<00:00, 31.28it/s]

Retrieval complete! Results saved to c:\Users\user\Desktop\workdir\ORDER_EMNLP2026\data\retrieval\MH_bm25_results.jsonl


In [5]:
# Evaluate all questions
df_result = pd.DataFrame(read_jsonl_basic(retrieval_results_file))
result = evaluate_and_print_metrics(df_result, k_values=[1, 2, 3, 5, 10, 30])


EVALUATION RESULTS FOR: DATASET

Computing recall metrics...


100%|██████████| 885/885 [00:00<00:00, 14244.50it/s]



Recall Metrics for 885 questions:
  recall@1: 0.134
  recall@2: 0.187
  recall@3: 0.231
  recall@5: 0.293
  recall@10: 0.378
  recall@30: 0.378


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 13157.75it/s]



Accuracy Metrics for 885 questions:
  accuracy@1: 0.000
  accuracy@2: 0.014
  accuracy@3: 0.043
  accuracy@5: 0.080
  accuracy@10: 0.166
  accuracy@30: 0.166


Computing MRR: 100%|██████████| 885/885 [00:00<00:00, 8912.39it/s]


MRR Metrics for 885 questions:
  mrr@1: 0.134
  mrr@2: 0.160
  mrr@3: 0.175
  mrr@5: 0.189
  mrr@10: 0.200
  mrr@30: 0.200


In [6]:
# Evaluate only newspaper × newspaper questions (source_2 is not Les Débats)
dft = df_result.loc[df_result["source_2"] != "Les Débats"]
result_newspapers = evaluate_and_print_metrics(dft)


EVALUATION RESULTS FOR: DATASET

Computing recall metrics...


100%|██████████| 314/314 [00:00<00:00, 17443.40it/s]



Recall Metrics for 314 questions:
  recall@2: 0.029
  recall@3: 0.046
  recall@5: 0.083
  recall@10: 0.131


Computing Accuracy: 100%|██████████| 314/314 [00:00<00:00, 22431.73it/s]



Accuracy Metrics for 314 questions:
  accuracy@2: 0.003
  accuracy@3: 0.010
  accuracy@5: 0.010
  accuracy@10: 0.032


Computing MRR: 100%|██████████| 314/314 [00:00<00:00, 5103.17it/s]


MRR Metrics for 314 questions:
  mrr@2: 0.024
  mrr@3: 0.030
  mrr@5: 0.038
  mrr@10: 0.044


In [7]:
# Evaluate only newspaper × Les Débats questions
dft = df_result.loc[df_result["source_2"] == "Les Débats"]
result_debats = evaluate_and_print_metrics(dft)


EVALUATION RESULTS FOR: DATASET

Computing recall metrics...


100%|██████████| 571/571 [00:00<00:00, 17309.54it/s]



Recall Metrics for 571 questions:
  recall@2: 0.274
  recall@3: 0.333
  recall@5: 0.409
  recall@10: 0.514


Computing Accuracy: 100%|██████████| 571/571 [00:00<00:00, 15221.38it/s]



Accuracy Metrics for 571 questions:
  accuracy@2: 0.019
  accuracy@3: 0.061
  accuracy@5: 0.119
  accuracy@10: 0.240


Computing MRR: 100%|██████████| 571/571 [00:00<00:00, 8593.03it/s]


MRR Metrics for 571 questions:
  mrr@2: 0.236
  mrr@3: 0.255
  mrr@5: 0.272
  mrr@10: 0.286
